# Xây dựng và phân tích mô hình K-means Custom trên bộ dữ liệu Wholesale Customers

Notebook này trình bày toàn bộ quy trình xử lý bài toán **phân cụm khách hàng** với mô hình **K-means tự xây dựng**.

Nội dung chính gồm:

- đọc và kiểm tra dữ liệu
- tiền xử lý dữ liệu phù hợp cho K-means
- trực quan hóa phân phối đặc trưng
- xây dựng mô hình **K-means custom**
- sử dụng **Elbow Method** để tìm số cụm `K` phù hợp
- huấn luyện mô hình cuối cùng
- trực quan hóa cụm bằng **PCA**
- phân tích đặc trưng trung bình của từng cụm

> Ghi chú: Bộ dữ liệu Wholesale Customers thường được dùng cho bài toán **Customer Segmentation**.  
> Mỗi dòng là một khách hàng, các cột thể hiện mức chi tiêu hằng năm theo từng nhóm hàng.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans


## 1. Đọc dữ liệu

Cell dưới đây hỗ trợ đọc dữ liệu từ nhiều vị trí thường gặp:

- file cùng thư mục notebook
- thư mục Kaggle Input
- tên file phổ biến của bộ dữ liệu

Bạn chỉ cần bảo đảm file CSV tồn tại ở một trong các đường dẫn gợi ý.

In [ ]:
candidate_paths = [
    "Wholesale customers data.csv",
    "wholesale_customers.csv",
    "Wholesale_Customers.csv",
    "/kaggle/input/wholesale-customers-data/Wholesale customers data.csv",
    "/kaggle/input/wholesale-customers/Wholesale customers data.csv",
    "/kaggle/input/wholesale-customers-dataset/Wholesale customers data.csv",
]

data_path = None
for path in candidate_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "Không tìm thấy file dữ liệu. Hãy đặt file CSV vào cùng thư mục notebook "
        "hoặc chỉnh lại danh sách candidate_paths."
    )

df = pd.read_csv(data_path)
print("Đường dẫn dữ liệu:", data_path)
print("Kích thước dữ liệu:", df.shape)
df.head()

## 2. Khám phá dữ liệu ban đầu

Ta kiểm tra:

- tên cột
- kiểu dữ liệu
- số lượng giá trị thiếu
- thống kê mô tả

In [ ]:
print("Các cột trong dữ liệu:")
print(df.columns.tolist())
print()

print("Thông tin dữ liệu:")
display(df.info())

print("Số lượng giá trị thiếu:")
display(df.isnull().sum().to_frame("missing_count"))

print("Thống kê mô tả:")
display(df.describe().T)

## 3. Ý nghĩa các nhóm đặc trưng

Bộ dữ liệu thường có các cột sau:

- `Channel`: loại khách hàng
- `Region`: khu vực
- `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen`: mức chi tiêu theo nhóm hàng

Trong bài toán K-means, ta thường ưu tiên các đặc trưng **hành vi chi tiêu liên tục**.  
Hai cột `Channel` và `Region` là biến mã hóa dạng phân loại nên sẽ được xử lý riêng.

## 4. Làm sạch và lựa chọn đặc trưng

### Hướng xử lý
- giữ lại các cột chi tiêu liên tục cho K-means
- loại `Channel`, `Region` khỏi tập đặc trưng phân cụm
- sau khi phân cụm xong có thể dùng lại `Channel`, `Region` để phân tích diễn giải cụm

In [ ]:
meta_cols = [col for col in ["Channel", "Region"] if col in df.columns]
spending_cols = [col for col in df.columns if col not in meta_cols]

print("Cột mô tả (không đưa trực tiếp vào K-means):", meta_cols)
print("Cột chi tiêu dùng để phân cụm:", spending_cols)

X_raw = df[spending_cols].copy()
display(X_raw.head())

## 5. Trực quan hóa dữ liệu trước tiền xử lý

Dữ liệu chi tiêu thường có phân phối lệch phải khá mạnh.  
Ta xem nhanh histogram để quan sát độ lệch của từng đặc trưng.

In [ ]:
num_cols = X_raw.columns.tolist()

n_cols = 3
n_rows = int(np.ceil(len(num_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(num_cols):
    axes[i].hist(X_raw[col], bins=25, edgecolor="black")
    axes[i].set_title(f"Histogram - {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## 6. Tiền xử lý dữ liệu

Với K-means, khoảng cách Euclid rất nhạy với:

- dữ liệu lệch mạnh
- chênh lệch thang đo giữa các đặc trưng

Do đó ta sẽ:

1. áp dụng `log1p` để giảm độ lệch
2. chuẩn hóa dữ liệu bằng z-score

In [ ]:
X_log = np.log1p(X_raw)

class StandardScalerCustom:
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_[self.std_ == 0] = 1.0
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

scaler = StandardScalerCustom()
X_scaled = scaler.fit_transform(X_log.values)

print("Shape sau tiền xử lý:", X_scaled.shape)
print("5 dòng đầu sau log + scale:")
pd.DataFrame(X_scaled, columns=spending_cols).head()

### 6.1. So sánh nhanh trước và sau biến đổi log

Biểu đồ dưới đây giúp quan sát dữ liệu sau `log1p`, thường sẽ bớt lệch hơn đáng kể.

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(num_cols):
    axes[i].hist(X_log[col], bins=25, edgecolor="black")
    axes[i].set_title(f"Histogram sau log1p - {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## 7. Xây dựng mô hình K-means custom

Ta tự cài đặt các thành phần cốt lõi:

- khởi tạo centroid ban đầu
- gán điểm vào centroid gần nhất
- cập nhật centroid theo trung bình cụm
- tính **WCSS** (Within-Cluster Sum of Squares)
- kiểm tra hội tụ

In [ ]:
class CustomKMeans:
    def __init__(self, n_clusters=3, max_iters=100, tol=1e-4, random_state=42):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.tol = tol
        self.random_state = random_state

        self.centroids = None
        self.labels_ = None
        self.inertia_ = None
        self.history_ = []

    def _initialize_centroids(self, X):
        rng = np.random.default_rng(self.random_state)
        indices = rng.choice(len(X), size=self.n_clusters, replace=False)
        return X[indices].copy()

    def _compute_distances(self, X, centroids):
        return np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)

    def _assign_labels(self, X, centroids):
        distances = self._compute_distances(X, centroids)
        return np.argmin(distances, axis=1)

    def _update_centroids(self, X, labels, old_centroids):
        new_centroids = []
        for i in range(self.n_clusters):
            cluster_points = X[labels == i]
            if len(cluster_points) == 0:
                new_centroids.append(old_centroids[i])
            else:
                new_centroids.append(cluster_points.mean(axis=0))
        return np.array(new_centroids)

    def _compute_wcss(self, X, labels, centroids):
        wcss = 0.0
        for i in range(self.n_clusters):
            cluster_points = X[labels == i]
            if len(cluster_points) > 0:
                wcss += np.sum((cluster_points - centroids[i]) ** 2)
        return float(wcss)

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        centroids = self._initialize_centroids(X)

        for iteration in range(self.max_iters):
            labels = self._assign_labels(X, centroids)
            new_centroids = self._update_centroids(X, labels, centroids)

            shift = np.linalg.norm(new_centroids - centroids)
            wcss = self._compute_wcss(X, labels, new_centroids)

            self.history_.append({
                "iteration": iteration + 1,
                "shift": shift,
                "wcss": wcss
            })

            if shift < self.tol:
                centroids = new_centroids
                break

            centroids = new_centroids

        self.centroids = centroids
        self.labels_ = self._assign_labels(X, self.centroids)
        self.inertia_ = self._compute_wcss(X, self.labels_, self.centroids)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return self._assign_labels(X, self.centroids)

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_

## 8. Chạy thử mô hình với một giá trị `K` mẫu

Phần này chỉ để kiểm tra mô hình custom hoạt động đúng trước khi đi vào Elbow Method.

In [ ]:
sample_k = 3
sample_model = CustomKMeans(n_clusters=sample_k, max_iters=100, tol=1e-4, random_state=42)
sample_labels = sample_model.fit_predict(X_scaled)

print("K =", sample_k)
print("WCSS =", sample_model.inertia_)
print("Centroids shape =", sample_model.centroids.shape)

history_df = pd.DataFrame(sample_model.history_)
display(history_df.head())
display(history_df.tail())

## 9. Elbow Method để tìm `K` phù hợp

Ta thử nhiều giá trị `K` và ghi lại **WCSS**.  
Nguyên tắc:

- `K` tăng thì WCSS giảm
- chọn vị trí mà tốc độ giảm bắt đầu chậm lại rõ rệt
- đó là điểm "khuỷu tay" của đồ thị

In [ ]:
k_values = list(range(1, 11))
wcss_values = []

for k in k_values:
    model = CustomKMeans(n_clusters=k, max_iters=100, tol=1e-4, random_state=42)
    model.fit(X_scaled)
    wcss_values.append(model.inertia_)

elbow_df = pd.DataFrame({
    "K": k_values,
    "WCSS": wcss_values
})

display(elbow_df)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, wcss_values, marker="o")
plt.xticks(k_values)
plt.xlabel("Số cụm K")
plt.ylabel("WCSS")
plt.title("Elbow Method - Custom K-means")
plt.grid(True, alpha=0.3)
plt.show()

### 9.1. Gợi ý chọn `K`

Bạn hãy nhìn biểu đồ Elbow ở trên và chọn `K` tại vị trí mà WCSS bắt đầu giảm chậm lại.

> Với bộ dữ liệu Wholesale Customers, `K = 3`, `4` hoặc `5` thường là các giá trị đáng cân nhắc.  
> Bạn có thể thay đổi `best_k` ở cell tiếp theo sau khi quan sát biểu đồ.

In [ ]:
best_k = 3   # Hãy chỉnh lại nếu bạn thấy điểm gãy phù hợp hơn
print("Giá trị K đang chọn:", best_k)

## 10. Huấn luyện mô hình cuối cùng với `K` đã chọn

In [ ]:
final_model = CustomKMeans(n_clusters=best_k, max_iters=100, tol=1e-4, random_state=42)
final_labels = final_model.fit_predict(X_scaled)

print("WCSS cuối cùng:", final_model.inertia_)
print("Số điểm trong từng cụm:")
print(pd.Series(final_labels).value_counts().sort_index())

## 10.1. So sánh với mô hình `KMeans` của sklearn

Để kiểm tra mô hình custom có hoạt động hợp lý hay không, ta huấn luyện thêm mô hình `KMeans` từ `sklearn` với cùng số cụm `K`.

Ta sẽ so sánh:
- **WCSS / Inertia**
- **kích thước cụm**
- **tọa độ centroid**
- **phân bố cụm trên không gian PCA 2 chiều**

> Lưu ý: nhãn cụm của K-means có thể bị **đảo thứ tự** giữa hai mô hình. Vì vậy khi so sánh, ta quan tâm nhiều hơn đến chất lượng phân cụm và vị trí cụm, không chỉ nhìn số nhãn `0, 1, 2, ...`.


In [ ]:
sklearn_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10,
    max_iter=300
)

sklearn_labels = sklearn_model.fit_predict(X_scaled)

print("WCSS custom:", final_model.inertia_)
print("WCSS sklearn:", sklearn_model.inertia_)
print()
print("Số điểm trong từng cụm - custom:")
print(pd.Series(final_labels).value_counts().sort_index())
print()
print("Số điểm trong từng cụm - sklearn:")
print(pd.Series(sklearn_labels).value_counts().sort_index())


### 10.2. So sánh centroid giữa hai mô hình

Centroid của mô hình custom và sklearn có thể xuất hiện theo thứ tự khác nhau. Bảng dưới đây vẫn rất hữu ích để quan sát xem hai mô hình có tạo ra các tâm cụm gần giống nhau hay không.


In [ ]:
custom_centroids_df = pd.DataFrame(
    final_model.centroids,
    columns=spending_cols,
    index=[f"Custom Cluster {i}" for i in range(best_k)]
)

sklearn_centroids_df = pd.DataFrame(
    sklearn_model.cluster_centers_,
    columns=spending_cols,
    index=[f"Sklearn Cluster {i}" for i in range(best_k)]
)

print("Centroid của Custom K-means (trên dữ liệu đã scale):")
display(custom_centroids_df.round(4))

print("Centroid của Sklearn KMeans (trên dữ liệu đã scale):")
display(sklearn_centroids_df.round(4))


### 10.3. Trực quan hóa cụm của hai mô hình trên PCA 2 chiều

Biểu đồ dưới đây giúp quan sát trực quan xem hai mô hình có chia dữ liệu theo cấu trúc tương tự nhau hay không.


In [ ]:
pca_compare = PCA(n_components=2, random_state=42)
X_pca_compare = pca_compare.fit_transform(X_scaled)

custom_centroids_pca = pca_compare.transform(final_model.centroids)
sklearn_centroids_pca = pca_compare.transform(sklearn_model.cluster_centers_)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_pca_compare[:, 0], X_pca_compare[:, 1], c=final_labels, s=45, alpha=0.8)
axes[0].scatter(custom_centroids_pca[:, 0], custom_centroids_pca[:, 1], marker="X", s=220)
axes[0].set_title("Custom K-means")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_pca_compare[:, 0], X_pca_compare[:, 1], c=sklearn_labels, s=45, alpha=0.8)
axes[1].scatter(sklearn_centroids_pca[:, 0], sklearn_centroids_pca[:, 1], marker="X", s=220)
axes[1].set_title("Sklearn KMeans")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 10.4. Nhận xét nhanh về phần so sánh

Khi chạy notebook, bạn có thể nhận xét theo hướng sau:
- nếu **WCSS của custom gần với sklearn**, mô hình custom đang hoạt động tốt
- nếu biểu đồ PCA của hai mô hình cho cấu trúc cụm tương tự nhau, điều đó cho thấy cách cài đặt custom là hợp lý
- nếu thứ tự nhãn cụm khác nhau, đây là hiện tượng bình thường của K-means
- nếu chênh lệch WCSS quá lớn, bạn nên kiểm tra lại bước khởi tạo centroid, điều kiện hội tụ hoặc cách cập nhật centroid


## 11. Gắn nhãn cụm vào dữ liệu gốc

Bước này giúp ta dễ phân tích đặc trưng của từng cụm trên dữ liệu ban đầu.

In [ ]:
df_clustered = df.copy()
df_clustered["Cluster"] = final_labels

display(df_clustered.head())

## 12. Phân tích trung bình đặc trưng theo cụm

Ta xem giá trị trung bình của từng đặc trưng chi tiêu trong mỗi cụm để hiểu phong cách mua hàng.

In [ ]:
cluster_profile = df_clustered.groupby("Cluster")[spending_cols].mean().round(2)
display(cluster_profile)

cluster_size = df_clustered["Cluster"].value_counts().sort_index().to_frame("count")
display(cluster_size)

In [ ]:
cluster_profile.T.plot(kind="bar", figsize=(14, 7))
plt.title("Trung bình đặc trưng theo từng cụm")
plt.xlabel("Feature")
plt.ylabel("Mean value")
plt.xticks(rotation=45)
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

## 13. Phân tích `Channel` và `Region` theo cụm

Mặc dù không đưa `Channel` và `Region` trực tiếp vào K-means, ta vẫn có thể dùng chúng để **diễn giải** kết quả phân cụm.

In [ ]:
if "Channel" in df_clustered.columns:
    print("Phân bố Channel theo cụm:")
    display(pd.crosstab(df_clustered["Cluster"], df_clustered["Channel"]))

if "Region" in df_clustered.columns:
    print("Phân bố Region theo cụm:")
    display(pd.crosstab(df_clustered["Cluster"], df_clustered["Region"]))

## 14. Trực quan hóa cụm bằng PCA 2 chiều

Do dữ liệu gốc có nhiều chiều, ta dùng **PCA** để chiếu dữ liệu xuống 2 chiều và quan sát phân bố cụm một cách trực quan hơn.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

centroids_pca = pca.transform(final_model.centroids)

plt.figure(figsize=(9, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=final_labels, s=45, alpha=0.8)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], marker="X", s=250, edgecolors="black")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title(f"PCA Visualization of Clusters (K = {best_k})")
plt.grid(True, alpha=0.3)
plt.show()

## 15. Tỷ lệ phương sai giải thích của PCA

In [ ]:
explained_variance = pca.explained_variance_ratio_
print("Tỷ lệ phương sai giải thích của PC1 và PC2:", explained_variance)
print("Tổng phương sai giải thích:", explained_variance.sum())

## 16. Diễn giải kết quả cụm

Khi xem bảng trung bình đặc trưng theo cụm, bạn có thể diễn giải theo hướng:

- cụm có `Grocery`, `Milk`, `Detergents_Paper` cao: có thể nghiêng về nhóm khách hàng bán lẻ
- cụm có `Fresh`, `Frozen` cao: có thể nghiêng về nhóm khách hàng nhà hàng / khách sạn / dịch vụ ăn uống
- cụm có mức chi tiêu thấp ở hầu hết cột: có thể là nhóm khách hàng quy mô nhỏ

Đây là bước rất quan trọng vì bài toán clustering không có nhãn đúng/sai cố định, nên ý nghĩa của cụm cần được rút ra từ dữ liệu.

## 17. Kết luận

Notebook đã hoàn thành các bước:

- xử lý dữ liệu Wholesale Customers
- chuẩn hóa dữ liệu phù hợp với K-means
- xây dựng mô hình **K-means custom**
- áp dụng **Elbow Method** để tìm số cụm `K`
- so sánh mô hình custom với **KMeans của sklearn**
- huấn luyện mô hình cuối cùng
- trực quan hóa và diễn giải cụm khách hàng

Bạn có thể mở rộng thêm theo các hướng:

- thử nhiều cách khởi tạo centroid
- bổ sung đánh giá bằng Silhouette Score
- so sánh với `KMeans` của sklearn
- dùng PCA / t-SNE để trực quan hóa sâu hơn